# Does body temperature shape the cold-shock gene CIRBP across mammals?
Part 2b of the sleep-CIRBP project (Riddhika D).

**Why:** In a first test on 22 Ensembl mammals, CIRBP's codon efficiency did **not** rise with longevity. It tracked **body temperature** instead: cooler-bodied species had lower CIRBP GC3 and CAI. This notebook re-tests that on **hundreds of mammals from NCBI**, with proper controls.

| Test | Question |
|---|---|
| 1 | Does CIRBP's codon make-up (GC3, CAI, Arg-CG codons) change with **body temperature**? |
| 2 | Does **longevity** add anything once temperature is in the model? |
| 3 | Is it CIRBP-specific? Every test is repeated on ~65 other genes. CIRBP only counts if it beats most of them. |
| 4 | Is it just genome-wide GC drift? GC3 is measured **relative to each species' own background** (its median across ~65 genes). |
| 5 | Is it just family history? Models compare species **within the same mammalian order**, with errors clustered by family, plus a mixed-model check. |

**Run:** Runtime → Run all (about 20–40 min, mostly downloading). Copy the **SEND THIS TO CLAUDE** box at the end.

In [ ]:
!pip -q install biopython statsmodels 2>/dev/null | grep -v -i dependency || true
!test -x ./datasets || (wget -q -O datasets https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets && chmod +x datasets)
import os, io, re, json, time, zipfile, warnings, requests, itertools, subprocess
import numpy as np, pandas as pd
from scipy import stats
warnings.filterwarnings('ignore'); os.makedirs('ncbi', exist_ok=True); os.makedirs('data', exist_ok=True)
REPORT = []
def note(s): print(s); REPORT.append(str(s))
print(subprocess.run(['./datasets', 'version'], capture_output=True, text=True).stdout)

TARGET = ['CIRBP', 'RBM3']
# reference set for codon adaptation: highly expressed genes
REF = ['RPL3','RPL4','RPL5','RPL7','RPL8','RPL10','RPL11','RPL13','RPL18','RPL23',
       'RPS2','RPS3','RPS4X','RPS6','RPS8','RPS11','RPS14','RPS19','EEF1A1','EEF2','GAPDH','ACTB']
# control genes: other RNA-binding proteins, housekeeping genes and DNA-repair genes
CONTROL = ['HNRNPA1','HNRNPA2B1','HNRNPK','SRSF1','SRSF3','SRSF7','PABPC1','ELAVL1','YBX1','TARDBP',
           'FUS','G3BP1','KHDRBS1','PTBP1','SFPQ','NONO','HNRNPU','CSDE1','SYNCRIP','RBMX',
           'PSMA1','PSMB1','POLR2A','CCT2','HSPA8','HSP90AB1','PPIA','LDHA','PGK1','ALDOA',
           'TPI1','ENO1','UBC','TUBB','VCP','SNRPB','NPM1','NCL','EIF4A1','EIF3A',
           'PCNA','XRCC5','XRCC6','PARP1','RAD51']
note(f'genes: {len(TARGET)} target, {len(REF)} reference, {len(CONTROL)} control')

## 1. Download coding sequences of every mammal ortholog (NCBI Datasets)
NCBI's ortholog sets cover several hundred mammal genomes. Each gene is saved in `ncbi/`, so a re-run skips finished genes.
For each species we keep the isoform whose length is closest to the gene's most common length across mammals (the conserved main isoform).

In [ ]:
def download(sym, tries=3):
    fn = f'ncbi/{sym}.zip'
    if os.path.exists(fn) and os.path.getsize(fn) > 1000: return fn
    for i in range(tries):
        r = subprocess.run(['./datasets', 'download', 'gene', 'symbol', sym, '--ortholog', '40674',
                            '--include', 'cds', '--filename', fn, '--no-progressbar'],
                           capture_output=True, text=True, timeout=900)
        if r.returncode == 0 and os.path.exists(fn): return fn
        print('  retry', sym, i + 1, (r.stderr or r.stdout)[-120:].strip()); time.sleep(10)
    print('  SKIPPED', sym); return None

def read_fasta(fh):
    h, s = None, []
    for line in io.TextIOWrapper(fh, encoding='utf-8'):
        line = line.strip()
        if line.startswith('>'):
            if h: yield h, ''.join(s)
            h, s = line[1:], []
        elif line: s.append(line)
    if h: yield h, ''.join(s)

def clean_cds(s):
    s = re.sub('[^ACGTN]', 'N', s.upper())
    if len(s) % 3: return None
    if s[-3:] in ('TAA', 'TAG', 'TGA'): s = s[:-3]
    return s if len(s) >= 150 else None

def parse(fn):
    z = zipfile.ZipFile(fn)
    gid2tax = {}
    for n in z.namelist():
        if n.endswith('data_report.jsonl'):
            for line in z.open(n):
                j = json.loads(line); gid2tax[str(j.get('geneId'))] = j.get('taxname')
    cand = {}
    for n in z.namelist():
        if not n.endswith('cds.fna'): continue
        for h, s in read_fasta(z.open(n)):
            org = re.search(r'\[organism=([^\]]+)\]', h); gid = re.search(r'\[GeneID=(\d+)\]', h)
            tax = org.group(1) if org else gid2tax.get(gid.group(1) if gid else '', None)
            s = clean_cds(s)
            if tax and s: cand.setdefault(tax, []).append(s)
    if not cand: return {}, None
    lens = pd.Series([len(s) // 3 for v in cand.values() for s in set(v)])
    L = int(lens.mode().iloc[0])                               # the conserved main isoform length
    out = {}
    for tax, v in cand.items():
        best = min(v, key=lambda s: abs(len(s) // 3 - L))
        if abs(len(best) // 3 - L) <= 0.15 * L: out[tax] = best
    return out, L

SEQ, MAINLEN = {}, {}
t0 = time.time()
for k, g in enumerate(TARGET + REF + CONTROL):
    fn = download(g)
    if fn:
        try: SEQ[g], MAINLEN[g] = parse(fn)
        except Exception as e: print('  parse failed', g, e); SEQ[g] = {}
    else: SEQ[g] = {}
    if k % 10 == 0: print(f'{k + 1}/{len(TARGET + REF + CONTROL)} genes, {time.time() - t0:.0f}s')
FAILED = [g for g, v in SEQ.items() if not v]
note(f'downloaded {len(SEQ) - len(FAILED)} genes; failed: {FAILED}')
note(f'CIRBP: {len(SEQ.get("CIRBP", {}))} species, main isoform {MAINLEN.get("CIRBP")} aa (human UniProt Q14011 = 172 aa); '
     f'RBM3: {len(SEQ.get("RBM3", {}))} species, {MAINLEN.get("RBM3")} aa')
note(f'median species per gene: {int(np.median([len(v) for v in SEQ.values() if v]))}')

## 2. Lifespan, body mass and body temperature (AnAge)
The **longevity residual** is how much longer a species lives than expected for its body mass. Body temperature comes from AnAge too.

In [ ]:
fn = 'data/anage.zip'
if not os.path.exists(fn):
    r = requests.get('https://genomics.senescence.info/species/dataset.zip', timeout=120); r.raise_for_status()
    open(fn, 'wb').write(r.content)
z = zipfile.ZipFile(fn); name = [n for n in z.namelist() if n.endswith('.txt')][0]
anage = pd.read_csv(z.open(name), sep='\t', encoding='latin-1')
anage = anage[anage['Class'] == 'Mammalia'].copy()
anage['species'] = (anage['Genus'] + '_' + anage['Species']).str.lower()
anage = anage.rename(columns={'Maximum longevity (yrs)': 'maxlon', 'Body mass (g)': 'mass', 'Temperature (K)': 'tempK'})
anage = anage[['species', 'Order', 'Family', 'Common name', 'maxlon', 'mass', 'tempK']].drop_duplicates('species')
note(f'AnAge mammals: {len(anage)}; with body temperature: {anage.tempK.notna().sum()}')

SYNONYM = {'physeter_catodon': 'physeter_macrocephalus', 'chlorocebus_sabaeus': 'chlorocebus_aethiops',
           'cebus_imitator': 'cebus_capucinus', 'nannospalax_galili': 'spalax_ehrenbergi',
           'cricetulus_griseus': 'cricetulus_barabensis', 'canis_lupus': 'canis_familiaris',
           'bos_indicus': 'bos_taurus', 'equus_asinus': 'equus_africanus', 'felis_silvestris': 'felis_catus',
           'ovis_aries': 'ovis_aries', 'sus_scrofa': 'sus_scrofa', 'mustela_putorius': 'mustela_putorius'}
have = set(anage.species)
def key(tax):
    s = '_'.join(tax.lower().replace('-', ' ').split()[:2])
    return s if s in have else SYNONYM.get(s, s)

sp = pd.DataFrame({'tax': sorted(SEQ.get('CIRBP', {}))})
sp['species'] = sp.tax.map(key)
sp = sp.merge(anage, on='species', how='left')
unmatched = sp.loc[sp.maxlon.isna(), 'tax'].tolist()
sp = sp.dropna(subset=['maxlon', 'mass']).drop_duplicates('species')
fit = np.polyfit(np.log10(sp.mass), np.log10(sp.maxlon), 1)
sp['lon_resid'] = np.log10(sp.maxlon) - np.polyval(fit, np.log10(sp.mass))
sp['tempC'] = sp.tempK - 273.15
note(f'CIRBP species matched to AnAge: {len(sp)} (unmatched {len(unmatched)}); orders: {sp.Order.nunique()}; '
     f'families: {sp.Family.nunique()}; with body temperature: {sp.tempK.notna().sum()}')
tt = sp.dropna(subset=['tempC']).sort_values('tempC')
note('coolest: ' + ', '.join(f'{r["Common name"]} {r.tempC:.1f}C' for _, r in tt.head(6).iterrows()))
note('warmest: ' + ', '.join(f'{r["Common name"]} {r.tempC:.1f}C' for _, r in tt.tail(6).iterrows()))
note(f'temperature vs longevity residual: rho={stats.spearmanr(tt.tempC, tt.lon_resid)[0]:+.2f} '
     f'(if strong, the two are confounded and need the joint model below)')
sp.to_csv('data/species.csv', index=False)

## 3. Codon measures
- **GC3**: GC at the third codon position. **GC3_rel** = gene GC3 minus the species' median GC3 across all ~65 other genes, so genome-wide GC drift cancels out.
- **CAI**: codon match to the species' own highly expressed genes (ribosomal proteins). **CAI_rel**: same idea, relative to the species' median CAI across the other genes.
- **Arg CG share**: fraction of arginine codons that are CGN (mutation-prone CpG codons).

In [ ]:
CODON_TABLE = {}
bases = 'TCAG'; aas = 'FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG'
for i, (a, b, c) in enumerate(itertools.product(bases, repeat=3)): CODON_TABLE[a + b + c] = aas[i]
SYN = {}
for c, aa in CODON_TABLE.items(): SYN.setdefault(aa, []).append(c)
def codons(s): return [s[i:i+3] for i in range(0, len(s) - 2, 3) if 'N' not in s[i:i+3]]
def weights(refs):
    cnt = {}
    for s in refs:
        for c in codons(s): cnt[c] = cnt.get(c, 0) + 1
    w = {}
    for aa, cs in SYN.items():
        if aa in '*MW': continue
        mx = max(cnt.get(c, 0) for c in cs)
        for c in cs: w[c] = max(cnt.get(c, 0), 0.5) / max(mx, 1)
    return w
def cai(s, w):
    v = [np.log(w[c]) for c in codons(s) if c in w]; return float(np.exp(np.mean(v))) if v else np.nan
def gc3(s):
    cs = [c for c in codons(s) if CODON_TABLE.get(c, '*') not in '*MW']; return float(np.mean([c[2] in 'GC' for c in cs])) if cs else np.nan
def arg_cg(s):
    cs = [c for c in codons(s) if CODON_TABLE.get(c) == 'R']; return float(np.mean([c.startswith('CG') for c in cs])) if cs else np.nan

taxset = set(sp.tax)
W = {}
for t in taxset:
    refs = [SEQ[g][t] for g in REF if t in SEQ.get(g, {})]
    if len(refs) >= 10: W[t] = weights(refs)
note(f'species with >=10 reference genes for CAI: {len(W)} of {len(taxset)}')

rows = []
for g in TARGET + CONTROL + REF:
    for t, s in SEQ.get(g, {}).items():
        if t in taxset:
            rows.append(dict(gene=g, tax=t, gc3=gc3(s), arg_cg=arg_cg(s), cai=cai(s, W[t]) if t in W else np.nan))
M = pd.DataFrame(rows)
bg = M[M.gene.isin(CONTROL + REF)].groupby('tax')[['gc3', 'cai']].median().rename(columns={'gc3': 'bg_gc3', 'cai': 'bg_cai'})
M = M.merge(bg, on='tax', how='left')
M['gc3_rel'] = M.gc3 - M.bg_gc3
M['cai_rel'] = M.cai - M.bg_cai
M = M.merge(sp[['tax', 'Order', 'Family', 'Common name', 'lon_resid', 'tempC']], on='tax')
M.to_csv('data/codon_metrics.csv', index=False)
note(f'gene x species rows: {len(M)}')

## 4. Tests
For every gene and measure:
- **Joint model:** measure ~ temperature + longevity residual + mammalian order (fixed effect), errors clustered by family. Each coefficient is "the effect of one, holding the other fixed".
- **Longevity-only model** on all species (many lack temperature data).
- **Null:** where CIRBP's t-value ranks among the control genes. CIRBP only counts if |t| beats most controls **and** p < 0.05.

In [ ]:
def zs(x): return (x - x.mean()) / x.std()

def ols_cluster(d, ycol, xcols, order='Order', cluster='Family'):
    # OLS with mammalian order as a fixed effect and cluster-robust (by family) standard errors
    X = pd.concat([pd.Series(1.0, index=d.index, name='const'), d[xcols].astype(float),
                   pd.get_dummies(d[order], prefix='o', drop_first=True, dtype=float)], axis=1)
    X = X.loc[:, X.std() > 0].assign(const=1.0) if len(X.columns) > 1 else X
    y = d[ycol].astype(float).values; Xv = X.values
    XtX_inv = np.linalg.pinv(Xv.T @ Xv); beta = XtX_inv @ Xv.T @ y; u = y - Xv @ beta
    g = pd.factorize(d[cluster])[0]; G = g.max() + 1; N, K = Xv.shape
    meat = np.zeros((K, K))
    for k in range(G):
        s = Xv[g == k].T @ u[g == k]; meat += np.outer(s, s)
    V = XtX_inv @ meat @ XtX_inv * (G / max(G - 1, 1)) * ((N - 1) / max(N - K, 1))
    se = np.sqrt(np.clip(np.diag(V), 1e-300, None)); t = beta / se
    p = 2 * stats.t.sf(np.abs(t), df=max(G - 1, 1))
    idx = list(X.columns)
    return {c: (t[idx.index(c)], p[idx.index(c)]) for c in xcols}

def fit_gene(d, var):
    out = {}
    a = d.dropna(subset=[var, 'lon_resid'])
    a = a[a.Order.map(a.Order.value_counts()) >= 2]
    if len(a) >= 15 and a[var].std() > 0:
        r = ols_cluster(a.assign(y=zs(a[var]), lon=zs(a.lon_resid)), 'y', ['lon'])
        out.update(n_lon=len(a), t_lon_only=r['lon'][0], p_lon_only=r['lon'][1])
    b = a.dropna(subset=['tempC'])
    b = b[b.Order.map(b.Order.value_counts()) >= 2]
    if len(b) >= 15 and b[var].std() > 0:
        r = ols_cluster(b.assign(y=zs(b[var]), temp=zs(b.tempC), lon=zs(b.lon_resid)), 'y', ['temp', 'lon'])
        out.update(n_joint=len(b), t_temp=r['temp'][0], p_temp=r['temp'][1], t_lon=r['lon'][0], p_lon=r['lon'][1],
                   rho_temp_raw=stats.spearmanr(b[var], b.tempC)[0])
    return out or None

RES = {}
for var in ['gc3_rel', 'gc3', 'cai_rel', 'cai', 'arg_cg']:
    R = {g: fit_gene(M[M.gene == g], var) for g in TARGET + CONTROL}
    R = pd.DataFrame({g: v for g, v in R.items() if v}).T
    RES[var] = R; R.to_csv(f'data/tests_{var}.csv')
    if 't_temp' not in R: note(f'{var}: too few species with temperature'); continue
    ctrl = R.loc[[c for c in CONTROL if c in R.index]]
    for g in TARGET:
        if g not in R.index or pd.isna(R.loc[g].get('t_temp')): note(f'{var} {g}: too few species'); continue
        r = R.loc[g]
        pt = (ctrl.t_temp.abs() < abs(r.t_temp)).mean() * 100
        pl = (ctrl.t_lon.abs() < abs(r.t_lon)).mean() * 100
        pl1 = (ctrl.t_lon_only.abs() < abs(r.t_lon_only)).mean() * 100
        note(f'{var:7s} {g:5s} | n={int(r.n_joint)} | TEMPERATURE t={r.t_temp:+.2f} p={r.p_temp:.2g} (beats {pt:.0f}% of {len(ctrl)} controls; raw rho {r.rho_temp_raw:+.2f}) '
             f'| LONGEVITY given temp t={r.t_lon:+.2f} p={r.p_lon:.2g} (beats {pl:.0f}%) '
             f'| longevity alone n={int(r.n_lon)} t={r.t_lon_only:+.2f} p={r.p_lon_only:.2g} (beats {pl1:.0f}%)')

### Robustness: mixed model (order and family as random effects) for CIRBP

In [ ]:
try: import statsmodels.formula.api as smf
except Exception: smf = None
for var in ['gc3_rel', 'cai_rel']:
    d = M[(M.gene == 'CIRBP')].dropna(subset=[var, 'lon_resid', 'tempC'])
    try:
        d = d.assign(y=zs(d[var]), temp=zs(d.tempC), lon=zs(d.lon_resid))
        m = smf.mixedlm('y ~ temp + lon', d, groups=d.Order, vc_formula={'Family': '0 + C(Family)'}).fit(reml=True)
        note(f'mixed model CIRBP {var}: temp beta={m.params["temp"]:+.2f} p={m.pvalues["temp"]:.2g}; '
             f'longevity beta={m.params["lon"]:+.2f} p={m.pvalues["lon"]:.2g} (n={len(d)})')
    except Exception as e:
        note(f'mixed model CIRBP {var} failed: {type(e).__name__}')

# leave-one-order-out: does any single order drive the temperature effect?
d = M[M.gene == 'CIRBP'].dropna(subset=['gc3_rel', 'lon_resid', 'tempC'])
lo = []
for o in d.Order.value_counts().index[:6]:
    x = d[d.Order != o]; x = x[x.Order.map(x.Order.value_counts()) >= 2]
    try:
        r = ols_cluster(x.assign(y=zs(x.gc3_rel), temp=zs(x.tempC), lon=zs(x.lon_resid)), 'y', ['temp', 'lon'])
        lo.append(f'without {o} (n={len(x)}): t={r["temp"][0]:+.2f}')
    except Exception: pass
note('CIRBP gc3_rel temperature effect, leaving out one order at a time: ' + '; '.join(lo))

## 5. Figures

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
d = M[M.gene == 'CIRBP'].dropna(subset=['gc3_rel', 'tempC'])
for o, dd in d.groupby('Order'):
    ax[0].scatter(dd.tempC, dd.gc3_rel, s=18, label=o if len(dd) >= 5 else None)
for _, r in pd.concat([d.nsmallest(4, 'tempC'), d.nlargest(3, 'tempC')]).iterrows():
    ax[0].annotate(str(r['Common name'])[:16], (r.tempC, r.gc3_rel), fontsize=7)
ax[0].set_xlabel('Body temperature (°C)'); ax[0].set_ylabel('CIRBP GC3 relative to species background')
ax[0].legend(fontsize=7, ncol=2); ax[0].set_title('CIRBP codon GC vs body temperature')
R = RES['gc3_rel']; ctrl = R.loc[[c for c in CONTROL if c in R.index], 't_temp'].dropna()
ax[1].hist(ctrl, bins=20, color='lightgray', edgecolor='gray', label='control genes')
for g, c in [('CIRBP', 'crimson'), ('RBM3', 'navy')]:
    if g in R.index and pd.notna(R.loc[g, 't_temp']): ax[1].axvline(R.loc[g, 't_temp'], color=c, lw=2, label=g)
ax[1].set_xlabel('temperature effect (t-value), GC3_rel'); ax[1].legend(); ax[1].set_title('Is CIRBP special?')
plt.tight_layout(); plt.savefig('data/figure_temperature.png', dpi=150); plt.show()

## How to read the result
- **Temperature hypothesis supported:** CIRBP's temperature t-value has p < 0.05 in the joint model, beats **≥ 90%** of control genes, holds in the mixed model, and no single order drives it.
- **Longevity:** if longevity stays near zero once temperature is in the model, the earlier "longevity" pattern was temperature in disguise.
- **Not supported:** CIRBP behaves like the control genes. That closes Part 2 honestly.

In [ ]:
print('=' * 70); print('SEND THIS TO CLAUDE'); print('=' * 70); print('\n'.join(REPORT)); print('=' * 70)